In [10]:
#Imports
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


import pandas as pd
# see https://github.com/marcharper/python-ternary for installation
import numpy as np

import random

from ast import literal_eval
from src.model_funcs import run_em, tolerance_sort, max_posterior_label, sort_matrix
from src.plotting_funcs import  plot_ternary_axes, plot_ternary_bounds, CB_color_cycle

# Bayesian Mixture Model

**Goal**: Model the uncertainty expressed by multiple annotations
**Tool**: Multinomial Mixture Model
- There exists a latent ground truth, i.e. a "true label" for each image
- This latent ground truth is unambiguous but the annotator’s opinion about it is not
- We set up a distributional framework and model the observed annotations under the assumption that there exists a latent true label
- The parameters of the associated distributions allow us to analyse the sources of labelling uncertainty and we can estimate the latent labels through the posterior probabilities

Assume latent ground truth:

$Z^{(i)}\sim Multi(\boldsymbol{\pi}, 1)$ iid.  $\rightarrow$ prior

Given latent true class, the labellers' votes follow:

$Y^{(i)}|Z^{(i)}\sim Multi(\boldsymbol{\theta_{Z^{(i)}}}, J^i)$ iid. $\rightarrow$ likelihood

or explicitly:

$Y^{(i)}|(Z^{(i)} = airplane) \sim Multi(\boldsymbol{\theta_{airplane}}, J^i) \\
Y^{(i)}|(Z^{(i)} = automobile) \sim Multi(\boldsymbol{\theta_{automobile}}, J^i)\\
Y^{(i)}|(Z^{(i)} = bird) \sim Multi(\boldsymbol{\theta_{bird}}, J^i)\\
Y^{(i)}|(Z^{(i)} = cat) \sim Multi(\boldsymbol{\theta_{cat}}, J^i)\\
Y^{(i)}|(Z^{(i)} = deer) \sim Multi(\boldsymbol{\theta_{deer}}, J^i)\\
Y^{(i)}|(Z^{(i)} = dog) \sim Multi(\boldsymbol{\theta_{dog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = frog) \sim Multi(\boldsymbol{\theta_{frog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = horse) \sim Multi(\boldsymbol{\theta_{horse}}, J^i)\\
Y^{(i)}|(Z^{(i)} = ship) \sim Multi(\boldsymbol{\theta_{ship}}, J^i)\\
Y^{(i)}|(Z^{(i)} = truck) \sim Multi(\boldsymbol{\theta_{truck}}, J^i).$


Use Bayes Rule to model latent ground truth given votes:

$\tau^{(i)}_l = P(Z^{(i)}=l|Y^{(i)}) = \frac{P(Z^{(i)}=l) \cdot P(Y^{(i)}|Z^{(i)} = l)}{P(Y^{(i)})} = \frac{prior \cdot likelihood}{evidence}$



Apply Expectation Maximization (EM) algorithm, iteratively estimate latent ground truth and parameters

1. E-Step:
calculate posterior class probabilities $\tau^{(i)}_l$ given $\pi$ and $\theta_{Z^{(i)}}$, i.e., parameters of prior and likelihood
choose class with highest posterior as latent ground truth class $Z^{(i)}$

2. M-Step:
update $\pi$ and $\theta_{Z^{(i)}}$ given $\tau^{(i)}_l$, i.e.,  posterior class probabilities

Initial values:
- $\pi$ = (1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10)
- $\theta$ drawn from dirichlet with $\alpha$ = (20,20,20,20,20,20,20,20,20,20) (i.e., 2* number_of_observed_classes)

# 1. Run EM

In [35]:
# load data
df_cifar10h = pd.read_csv('../cifar-10h/data/Y_cifar10h.csv', index_col=0)
df_R_cifar10h = pd.read_csv('../cifar-10h/data/Y_R_cifar10h.csv', index_col=1)
#df_snli.old_labels = df_snli.old_labels.apply(literal_eval) # since quotes in list elements are escaped

In [38]:
df_R_cifar10h[df_R_cifar10h["Reaction"] == "Fast"]

,Reaction,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck,J
image_filename,,,,,,,,,,,,
abandoned_ship_s_000213.png,Fast,0,1,0,0,0,0,0,0,37,0,38
abandoned_ship_s_000260.png,Fast,0,0,0,0,0,0,0,0,42,1,43
abandoned_ship_s_000380.png,Fast,1,0,0,0,0,0,0,0,38,0,39
abandoned_ship_s_000635.png,Fast,0,0,0,0,0,0,0,0,43,0,43
abandoned_ship_s_000735.png,Fast,1,0,0,0,0,0,0,0,43,0,44
...,...,...,...,...,...,...,...,...,...,...,...,...
wrecker_s_002367.png,Fast,0,0,1,0,0,0,0,0,0,50,51
yosemite_toad_s_000002.png,Fast,0,0,9,1,0,0,19,0,0,0,29
yosemite_toad_s_000015.png,Fast,0,0,0,0,0,0,45,0,0,0,45


In [39]:
#extract relevant columns
cifar10h_one_hot = df_cifar10h[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
cifar10h_one_hot_arr = np.array(cifar10h_one_hot).astype(int)

# filter df_R_cifar10h for Reaction == Slow
Slow_cifar10h_one_hot = df_R_cifar10h[df_R_cifar10h['Reaction'] == 'Slow'][['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
Slow_cifar10h_one_hot_arr = np.array(Slow_cifar10h_one_hot).astype(int)
Fast_cifar10h_one_hot = df_R_cifar10h[df_R_cifar10h['Reaction'] == 'Fast'][['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
Fast_cifar10h_one_hot_arr = np.array(Fast_cifar10h_one_hot).astype(int)
# cifar10h_one_hot_arr_test = cifar10h_one_hot_arr[0:100]
#frequency of all votes
rel_freq = np.sum(cifar10h_one_hot_arr,axis=0)/(100*len(cifar10h_one_hot_arr))

#frequency of personal ground truth
#gt_freq = np.unique(df_cifar10h['ground_truth'],return_counts=True)
#as the labels are initially sorted alphabetically, we manually fix the order: E, N, C
#rel_freq_gt = gt_freq[1][[1,2,0]]/len(df_snli)

#frequency of majority vote label
#m_vote_freq = np.unique(df_snli['majority_label'], return_counts=True)
#as the labels are initially sorted alphabetically, we manually fix the order: E, N, C
#rel_freq_m_vote = m_vote_freq[1][[1,2,0]]/len(df_snli)

In [40]:
# run em
random.seed(12)
cifar10h_em = run_em(cifar10h_one_hot_arr, K=10)
cifar10h_em_slow = run_em(Slow_cifar10h_one_hot_arr, K=10)
cifar10h_em_fast = run_em(Fast_cifar10h_one_hot_arr, K=10)

/Users/Eugen/Desktop/LMU/SS24/BachelorThesis/label-variation-cifar10h/src/model_funcs.py:109: RuntimeWarning: divide by zero encountered in log
  loss += np.sum(weights * (np.log(pi[k]) + np.log(self._multinomial_prob(Y, theta[k]))))
/Users/Eugen/Desktop/LMU/SS24/BachelorThesis/label-variation-cifar10h/src/model_funcs.py:109: RuntimeWarning: invalid value encountered in multiply
  loss += np.sum(weights * (np.log(pi[k]) + np.log(self._multinomial_prob(Y, theta[k]))))
/Users/Eugen/Desktop/LMU/SS24/BachelorThesis/label-variation-cifar10h/src/model_funcs.py:110: RuntimeWarning: divide by zero encountered in log
  loss -= np.sum(weights * np.log(weights))
/Users/Eugen/Desktop/LMU/SS24/BachelorThesis/label-variation-cifar10h/src/model_funcs.py:110: RuntimeWarning: invalid value encountered in multiply
  loss -= np.sum(weights * np.log(weights))


In [41]:
theta_sorted = sort_matrix(cifar10h_em[2])
theta_sorted_slow = sort_matrix(cifar10h_em_slow[2])
theta_sorted_fast = sort_matrix(cifar10h_em_fast[2])

In [8]:
df_cifar10h['posterior'] = cifar10h_em[3].tolist()

In [42]:
theta_sorted

array([[9.67478995e-01, 7.19433375e-03, 4.89579634e-03, 1.47037261e-03,
        1.11613121e-03, 6.56416343e-04, 1.31283269e-03, 9.71496183e-04,
        1.27768367e-02, 2.12678909e-03],
       [1.74196354e-03, 9.78842531e-01, 6.27106874e-04, 1.20755857e-03,
        1.13808284e-03, 9.05821039e-04, 5.80654537e-04, 5.80654512e-04,
        9.29047244e-04, 1.34465801e-02],
       [2.41819088e-03, 6.75463678e-04, 9.72471688e-01, 5.03952060e-03,
        4.48433908e-03, 3.62047162e-03, 7.39853414e-03, 1.62114390e-03,
        1.62217190e-03, 6.48476189e-04],
       [1.07553096e-03, 7.17028591e-04, 5.99127381e-03, 9.56449492e-01,
        3.12439291e-03, 2.24590246e-02, 6.32688527e-03, 1.59927465e-03,
        9.88001852e-04, 1.26909518e-03],
       [1.25262816e-03, 8.15748164e-04, 5.37884392e-03, 4.36926932e-03,
        9.50674959e-01, 7.58270796e-03, 4.95242908e-03, 2.25560049e-02,
        1.36920285e-03, 1.04820656e-03],
       [6.84994286e-04, 5.53264605e-04, 2.78037443e-03, 1.76957722e-02,
   

In [43]:
theta_sorted_slow

array([[8.92582655e-01, 1.07839174e-02, 3.01286691e-02, 4.90971303e-03,
        1.85845445e-03, 3.19661084e-03, 6.92148485e-03, 2.38029956e-03,
        3.97992205e-02, 7.43897563e-03],
       [2.50224266e-03, 9.26944371e-01, 9.52985546e-04, 4.04088769e-03,
        5.95617529e-04, 4.76489600e-04, 9.57347974e-04, 2.38407495e-04,
        1.31011902e-03, 6.19815311e-02],
       [6.29314898e-03, 1.82559340e-03, 8.80562944e-01, 2.54726761e-02,
        2.33218920e-02, 2.30542259e-02, 2.47047301e-02, 9.50882900e-03,
        4.20275794e-03, 1.05320242e-03],
       [1.94674792e-03, 2.24858916e-03, 3.43960511e-02, 8.17014185e-01,
        1.43958807e-02, 9.24988590e-02, 2.34244898e-02, 7.78263144e-03,
        4.15127214e-03, 2.14129399e-03],
       [2.13046873e-03, 1.21158583e-03, 2.26754321e-02, 1.86541527e-02,
        8.28621211e-01, 4.56602004e-02, 1.15293640e-02, 6.61202350e-02,
        2.06442356e-03, 1.33292688e-03],
       [6.82785577e-04, 2.53838083e-04, 1.01858284e-02, 6.07730565e-02,
   

In [44]:
theta_sorted_fast

array([[9.67478995e-01, 7.19433375e-03, 4.89579634e-03, 1.47037261e-03,
        1.11613121e-03, 6.56416343e-04, 1.31283269e-03, 9.71496183e-04,
        1.27768367e-02, 2.12678909e-03],
       [1.74196354e-03, 9.78842531e-01, 6.27106874e-04, 1.20755857e-03,
        1.13808284e-03, 9.05821039e-04, 5.80654537e-04, 5.80654512e-04,
        9.29047244e-04, 1.34465801e-02],
       [2.41819088e-03, 6.75463678e-04, 9.72471688e-01, 5.03952060e-03,
        4.48433908e-03, 3.62047162e-03, 7.39853414e-03, 1.62114390e-03,
        1.62217190e-03, 6.48476189e-04],
       [1.07553096e-03, 7.17028591e-04, 5.99127381e-03, 9.56449492e-01,
        3.12439291e-03, 2.24590246e-02, 6.32688527e-03, 1.59927465e-03,
        9.88001852e-04, 1.26909518e-03],
       [1.25262816e-03, 8.15748164e-04, 5.37884392e-03, 4.36926932e-03,
        9.50674959e-01, 7.58270796e-03, 4.95242908e-03, 2.25560049e-02,
        1.36920285e-03, 1.04820656e-03],
       [6.84994286e-04, 5.53264605e-04, 2.78037443e-03, 1.76957722e-02,
   

In [45]:
# Convert theta_matched, pi_matched, and tau_matched to DataFrames
df_theta = pd.DataFrame(cifar10h_em[2])
df_pi = pd.DataFrame(cifar10h_em[1])
df_tau = pd.DataFrame(cifar10h_em[3])

df_theta_slow = pd.DataFrame(cifar10h_em_slow[2])
df_pi_slow = pd.DataFrame(cifar10h_em_slow[1])
df_tau_slow = pd.DataFrame(cifar10h_em_slow[3])

df_theta_fast = pd.DataFrame(cifar10h_em_fast[2])
df_pi_fast = pd.DataFrame(cifar10h_em_fast[1])
df_tau_fast = pd.DataFrame(cifar10h_em_fast[3])

# Export DataFrames to CSV files
df_theta.to_csv('../cifar-10h/data/theta.csv', index=False)
df_pi.to_csv('../cifar-10h/data/pi.csv', index=False)
df_tau.to_csv('../cifar-10h/data/tau.csv', index=False)

df_theta_slow.to_csv('../cifar-10h/data/theta_slow.csv', index=False)
df_pi_slow.to_csv('../cifar-10h/data/pi_slow.csv', index=False)
df_tau_slow.to_csv('../cifar-10h/data/tau_slow.csv', index=False)

df_theta_fast.to_csv('../cifar-10h/data/theta_fast.csv', index=False)
df_pi_fast.to_csv('../cifar-10h/data/pi_fast.csv', index=False)
df_tau_fast.to_csv('../cifar-10h/data/tau_fast.csv', index=False)